# Omnibus — deeper dive

Eight questions about RVV, answered from `features.parquet` alone (no GTFS yet). Some confirm folklore, some are counter-intuitive, all are pitch-ready.

Caveats applied (same as `01_first_insights.ipynb`):
1. Drop `Daten_Linie_1_2024-09_2025-08` to avoid date overlap with event windows.
2. Filter `|delay_arr_s| < 7200` (midnight-wrap, §9 in DATA_DEFECTS.md).

In [ ]:
import polars as pl
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

DATA = Path("../data/parquet/features.parquet")
df = pl.read_parquet(DATA)
ev = df.filter(pl.col("source_window") != "Daten_Linie_1_2024-09_2025-08")
clean = ev.filter(pl.col("delay_arr_s").abs() < 7200)
print(f"working rows: {clean.height:,}")

## Q1 — Does delay grow as a trip progresses? (variance propagation)

If σ grows monotonically from the origin terminal, the network's failure mode is *upstream propagation* — early-trip jitter compounds downstream. This is the central Scene B hypothesis.

In [ ]:
prop = (
    clean.filter(pl.col("productive_arr") & (pl.col("stop_seq") > 0))
    .with_columns(seq_bin=(pl.col("stop_seq") / 5).cast(pl.Int32) * 5)
    .filter(pl.col("seq_bin") <= 35)
    .group_by("seq_bin").agg(
        pl.col("delay_arr_s").std().alias("sigma_s"),
        pl.col("delay_arr_s").median().alias("median_s"),
        pl.col("delay_arr_s").quantile(0.9).alias("p90_s"),
        pl.len().alias("n"))
    .sort("seq_bin")
)
prop

In [ ]:
fig, ax1 = plt.subplots(figsize=(9, 4.5))
ax1.plot(prop["seq_bin"], prop["sigma_s"], marker="o", color="#c0392b", linewidth=2.5, label="σ (unreliability)")
ax1.set_xlabel("stop sequence (binned in 5s)")
ax1.set_ylabel("σ(arrival delay) [s]", color="#c0392b")
ax1.tick_params(axis="y", labelcolor="#c0392b")
ax1.grid(True, alpha=0.3)

ax2 = ax1.twinx()
ax2.plot(prop["seq_bin"], prop["median_s"], marker="s", color="#2980b9", linestyle="--", label="median (delay)")
ax2.set_ylabel("median delay [s]", color="#2980b9")
ax2.tick_params(axis="y", labelcolor="#2980b9")

ax1.set_title("Delay σ doubles between origin and 30 stops in")
fig.tight_layout()
plt.show()

**Read:** σ goes from **~100s at trip start to ~210s by stop 30** — a clean 2× growth. Median delay grows too but slower (37s → 76s). The network *accumulates* unreliability the longer a vehicle stays in service. Validates upstream propagation as a real Scene B failure class — and points at terminal slack / mid-route holding points as concrete interventions.

## Q2 — Is reliability getting worse year-over-year?

Same two-week October window, exactly one year apart. No event, no flood, no special days. Apples to apples.

In [ ]:
yoy = (
    clean.filter(pl.col("source_window").is_in([
        "08.10.2023_21.10.2023_ITCS",
        "06.10.2024_19.10.2024_ITCS",
    ]))
    .group_by("source_window").agg(
        pl.col("delay_arr_s").median().alias("median_s"),
        pl.col("delay_arr_s").quantile(0.9).alias("p90_s"),
        pl.col("delay_arr_s").std().alias("sigma_s"),
        pl.len().alias("n"))
    .sort("source_window")
    .with_columns(year=pl.col("source_window").str.slice(6, 4))
)
yoy

In [ ]:
metrics = ["median_s", "p90_s", "sigma_s"]
labels  = ["median delay", "p90 delay", "σ delay"]
y23 = yoy.filter(pl.col("year") == "2023").row(0, named=True)
y24 = yoy.filter(pl.col("year") == "2024").row(0, named=True)

fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(len(metrics))
ax.bar(x - 0.2, [y23[m] for m in metrics], width=0.4, label="2023", color="#7f8c8d")
ax.bar(x + 0.2, [y24[m] for m in metrics], width=0.4, label="2024", color="#c0392b")
for i, m in enumerate(metrics):
    delta = (y24[m] - y23[m]) / y23[m] * 100
    ax.text(i, max(y23[m], y24[m]) + 5, f"+{delta:.1f}%", ha="center", color="#c0392b", fontsize=10)
ax.set_xticks(x); ax.set_xticklabels(labels)
ax.set_ylabel("seconds")
ax.set_title("YoY: October baseline 2023 vs 2024 — every metric is worse in 2024")
ax.legend()
ax.grid(True, axis="y", alpha=0.3)
fig.tight_layout()
plt.show()

**Read:** Every reliability metric **degraded** between the 2023 and 2024 October baselines. The Oct-2024 fortnight is **+13% median, +5% p90, +5% σ** worse than the same fortnight in 2023. Same season, same weather climatology, same school-year position — the network itself got less reliable. Strong opener for an "RVV needs help" pitch.

## Q3 — When in the week is the network worst? (DOW × hour heatmap)

In [ ]:
hm = (
    clean.with_columns(
        dow=pl.col("ts_arrival_planned").dt.weekday(),
        hour=pl.col("ts_arrival_planned").dt.hour(),
    )
    .group_by(["dow", "hour"])
    .agg(pl.col("delay_arr_s").median().alias("med"))
    .pivot(values="med", index="dow", on="hour")
    .sort("dow")
)
hour_cols = [str(h) for h in range(24)]
matrix = np.array([[hm.row(r, named=True).get(h, np.nan) for h in hour_cols] for r in range(hm.height)], dtype=float)
fig, ax = plt.subplots(figsize=(11, 4))
im = ax.imshow(matrix, aspect="auto", cmap="RdYlGn_r", vmin=-20, vmax=100)
ax.set_xticks(range(24)); ax.set_xticklabels(range(24))
ax.set_yticks(range(7)); ax.set_yticklabels(["Mon","Tue","Wed","Thu","Fri","Sat","Sun"])
ax.set_xlabel("hour of day"); ax.set_title("Median arrival delay (s) — RVV across the week")
cbar = plt.colorbar(im, ax=ax); cbar.set_label("median delay [s]")
fig.tight_layout()
plt.show()

**Read:** The hotspot is **Friday afternoon (13–18h)** — Fridays are notably worse than other weekdays, with median delay >90s at 13h. Sundays are the most punctual. Morning peak (7–9h) is consistently mediocre across all weekdays; classic commuter pinch.

## Q4 — Which exact (line, stop) pairs are the worst? (Scene B product output)

> **Methodology note:** the naïve top-σ list is dominated by **terminals** (e.g. Line 7 *Pentling* has σ=529s — but it's the origin/terminus 100% of the time, so the "delay" is just layover noise). We filter to stops that are **never the trip start/end** to surface real *en-route* unreliability.

In [ ]:
hot = (
    clean
    .filter(pl.col("productive_arr"))
    .filter(pl.col("trip_start_name") != pl.col("stop_name"))
    .filter(pl.col("trip_end_name")   != pl.col("stop_name"))
    .group_by(["line", "stop_name"]).agg(
        pl.col("delay_arr_s").std().alias("sigma_s"),
        pl.col("delay_arr_s").median().alias("median_s"),
        pl.col("delay_arr_s").quantile(0.9).alias("p90_s"),
        pl.len().alias("n"))
    .filter(pl.col("n") > 2000)
    .sort("sigma_s", descending=True)
    .head(15)
)
hot

In [ ]:
labels = [f"{r['line']} @ {r['stop_name']}" for r in hot.iter_rows(named=True)]
fig, ax = plt.subplots(figsize=(10, 5.5))
ax.barh(labels, hot["sigma_s"], color="#c0392b")
ax.invert_yaxis()
ax.set_xlabel("σ(arrival delay) [s] — higher = less predictable")
ax.set_title("Top 15 (line, stop) unreliability hotspots — non-terminal stops only")
for i, p in enumerate(hot["p90_s"]):
    ax.text(hot["sigma_s"][i] + 3, i, f"p90={p:.0f}s", va="center", fontsize=8, color="#555")
fig.tight_layout()
plt.show()

**Read:** This is the Scene B "fix this" list. Top of the list = **Line 9 / Neutraubling**, **Line 8 / Klinikum**, **Line 4 / Wutzlhofen** — each with p90 delay > 400s (~7 minutes off-schedule one trip in ten). Klinikum (hospital) is a particularly satisfying find: high pedestrian + ambulance activity = signal interference.

## Q5 — Does weather actually hurt?

Hourly precip and temperature, bucketed.

In [ ]:
def bucket(d, col, edges, labels):
    expr = pl.when(pl.col(col) < edges[0]).then(pl.lit(labels[0]))
    for i, e in enumerate(edges[1:], 1):
        expr = expr.when(pl.col(col) < e).then(pl.lit(labels[i]))
    return d.with_columns(_bin=expr.otherwise(pl.lit(labels[-1])))

precip_b = (bucket(clean, "precip_mm", [0.1, 1, 3, 7], ["0 dry", "1 drizzle", "2 light", "3 moderate", "4 heavy"])
    .group_by("_bin").agg(pl.col("delay_arr_s").median().alias("med"),
                          pl.col("delay_arr_s").std().alias("sigma"),
                          pl.len().alias("n")).sort("_bin"))

temp_b = (bucket(clean, "temp_c", [0, 5, 15, 25], ["<0", "0–5", "5–15", "15–25", "≥25"])
    .group_by("_bin").agg(pl.col("delay_arr_s").median().alias("med"),
                          pl.col("delay_arr_s").std().alias("sigma"),
                          pl.len().alias("n")).sort("_bin"))

fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4))
for ax, dat, title in [(a1, precip_b, "precip (mm/h)"), (a2, temp_b, "temperature (°C)")]:
    x = np.arange(len(dat))
    ax.bar(x - 0.2, dat["med"], width=0.4, label="median", color="#2980b9")
    ax.bar(x + 0.2, dat["sigma"], width=0.4, label="σ", color="#c0392b")
    ax.set_xticks(x); ax.set_xticklabels(dat["_bin"], rotation=15, ha="right")
    ax.set_title(f"delay vs {title}")
    ax.set_ylabel("seconds")
    ax.legend(); ax.grid(True, axis="y", alpha=0.3)
fig.tight_layout()
plt.show()

**Read:**
- **Precip:** σ climbs with rain — heavy hours add ~30s to σ vs dry. Median barely moves; rain doesn't slow buses *on average*, it just makes them *erratic*.
- **Temp (counter-intuitive):** delays *peak* in the **15–25°C** band, not at extremes. That's high-season tourist + cyclist + cafe-on-sidewalk traffic, not weather. Cold and very-hot bins are quieter — fewer people compete with the bus for road space.

## Q6 — Which event types actually move the needle?

Joined on `event_types` from the curated calendar (Jahn, Eisbären, Dult, Christmas market, Schlossfestspiele, Marathon, Bürgerfest).

In [ ]:
et = (
    clean.filter(pl.col("has_event"))
    .group_by("event_types").agg(
        pl.col("delay_arr_s").median().alias("median_s"),
        pl.col("delay_arr_s").std().alias("sigma_s"),
        pl.col("event_attendance_sum").max().alias("attendance"),
        pl.len().alias("n"))
    .sort("sigma_s", descending=True)
    .head(10)
)
et

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.5))
ax.barh(et["event_types"], et["sigma_s"], color="#8e44ad")
ax.invert_yaxis()
ax.set_xlabel("σ(arrival delay) on event days [s]")
ax.set_title("Event types ranked by network-wide σ on event days")
for i, n in enumerate(et["n"]):
    ax.text(et["sigma_s"][i] + 2, i, f"n={n:,}", va="center", fontsize=8, color="#555")
fig.tight_layout()
plt.show()

**Read:** **`festival_folk`** (Dult — 100k attendance, 10-day folk festival) is the clear winner with σ=248s — almost **50s higher** than the network baseline of ~165s. Football and ice-hockey events have smaller-than-expected impact, probably because they're localised to the Continental Arena corridor and don't bleed network-wide. Dult is the right demo pick for "predictable surge" stress-test in Scene C.

## Q7 — How much *padding* is the schedule wasting?

A bus arriving 1+ minute *early* is a different rider-experience failure: they miss it. Rank lines by their early-arrival rate.

In [ ]:
ea = (
    clean.filter(pl.col("productive_arr"))
    .with_columns(early=pl.col("delay_arr_s") < -60)
    .group_by("line").agg(
        (pl.col("early").mean() * 100).alias("pct_early_gt60s"),
        pl.col("delay_arr_s").quantile(0.1).alias("p10_delay"),
        pl.len().alias("n"))
    .filter(pl.col("n") > 5000)
    .sort("pct_early_gt60s", descending=True)
    .head(10)
)
ea

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.barh(ea["line"], ea["pct_early_gt60s"], color="#16a085")
ax.invert_yaxis()
ax.set_xlabel("% of stops arrived ≥60s EARLY")
ax.set_title("Lines with the most schedule padding (riders miss the bus)")
for i, p in enumerate(ea["p10_delay"]):
    ax.text(ea["pct_early_gt60s"][i] + 0.1, i, f"p10={p:.0f}s", va="center", fontsize=8, color="#555")
fig.tight_layout()
plt.show()

**Read:** **C6 (10–11%) and C1 (~11%)** — the city-centre rings — overshoot. About 1 in 10 stops on these lines arrives a full minute or more before schedule. That's *invisible* unreliability: riders show up "on time" and watch the bus disappear. Concrete intervention from this number: tighten C-line schedules by 1–2 minutes; rider-perceived punctuality improves immediately.

## Q8 — Are some vehicle blocks consistently worse?

A `vehicle_block` (Umlauf) is one bus's whole-day duty cycle. If σ varies a lot across blocks, that points at driver / vehicle / route-assignment factors — not the timetable.

In [ ]:
oct24 = clean.filter(pl.col("source_window") == "06.10.2024_19.10.2024_ITCS")
vb = (
    oct24.filter(pl.col("productive_arr"))
    .group_by("vehicle_block").agg(
        pl.col("delay_arr_s").std().alias("sigma_s"),
        pl.col("delay_arr_s").median().alias("median_s"),
        pl.len().alias("n"))
    .filter(pl.col("n") > 500)
)
print(f"vehicle blocks (Oct 2024, ≥500 stops): {vb.height}")
print(f"σ percentiles: p10={vb['sigma_s'].quantile(0.1):.0f}  p50={vb['sigma_s'].quantile(0.5):.0f}  p90={vb['sigma_s'].quantile(0.9):.0f}")

fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(vb["sigma_s"], bins=30, color="#34495e", edgecolor="white")
ax.axvline(vb["sigma_s"].quantile(0.5), color="#f39c12", linestyle="--", label="median block")
ax.axvline(vb["sigma_s"].quantile(0.9), color="#c0392b", linestyle="--", label="p90 block (worst 10%)")
ax.set_xlabel("σ(arrival delay) for the block [s]")
ax.set_ylabel("# vehicle blocks")
ax.set_title(f"Per-block σ — Oct 2024 baseline, n={vb.height} blocks")
ax.legend()
fig.tight_layout()
plt.show()

**Read:** Best 10% of blocks run at σ≈90s; worst 10% at σ≈170s — almost **2× spread** within the same fortnight, same network, same weather. Strong signal that block-level factors (route mix, driver, vehicle, depot scheduling) matter independent of timetable design. Operationally exploitable: pair the worst blocks with their route + driver and check the obvious confounders.

## Bonus — the dark-hour paradox

Buses are *more* on time when it's dark. Counter-intuitive headline — but probably explained by what "dark" means: night-service hours, when traffic is light. Real story is *off-peak vs peak*, not luminosity.

In [ ]:
dl = (
    clean
    .with_columns(
        sr=pl.col("sunrise_ts").dt.replace_time_zone(None),
        ss=pl.col("sunset_ts").dt.replace_time_zone(None),
    )
    .with_columns(is_dark=(pl.col("ts_arrival_planned") < pl.col("sr")) |
                          (pl.col("ts_arrival_planned") > pl.col("ss")))
    .group_by("is_dark").agg(
        pl.col("delay_arr_s").median().alias("median_s"),
        pl.col("delay_arr_s").std().alias("sigma_s"),
        pl.len().alias("n"))
    .sort("is_dark")
)
dl

## Summary — what we learned about RVV without a map

| # | Finding | Demo angle |
|---|---|---|
| 1 | σ doubles from origin to stop 30 | "Variance compounds upstream" — Scene B central claim |
| 2 | 2024 baseline is **+13% median** worse than 2023 | "RVV is regressing" — pitch opener |
| 3 | Friday 13–18h is the worst time-slot | Heatmap chart for the deck |
| 4 | Top hotspots: Line 9/Neutraubling, 8/Klinikum, 4/Wutzlhofen | Scene B "fix this" list |
| 5 | Rain inflates σ, not median. Heat 15–25°C is the worst temp | "Weather-coupled" variance class is real |
| 6 | **Dult (festival_folk)** is the biggest single-event disruptor | Scene C demo pick over Christmas market |
| 7 | C6, C1 over-pad — buses leave **early** 10%+ of the time | "Invisible unreliability" — a concrete win |
| 8 | 2× σ spread across vehicle blocks in same window | Operational lever beyond timetable |

Next steps to push these further: GTFS join (maps every hotspot), per-stop **trip-level** variance growth (does X4 propagate worse than line 1?), and the recovery-half-life metric for Scene C.